# Class of Buildings

This class contains both the buildings and the special tiles castle and mine.

The castle (dark green) allows us to immediately play another action of our choosing (as if we had rolled any number).<br>
The mine provide at the end of each phase 1 silverling.<br>
Buildings (beige) are of 8 types. Each can be placed at most one time in each cities (connected beige tiles) and provide an advantage right after being built.

Pour être en accord avec ce qu'à fait Emilie, je vais faire de la même manière avec une liste de tous les batîments et leur fonction associée.

L'idée pour chaque action, c'est qu'on aura les attributs qu'il faut pour faire fonctionner le tout (c'est ce que j'ai compris de ton code Emilie).
Du coup pour le *castle* par exemple, on aura un compte d'actions et la tile *castle* nous permet d'avoir une action de plus et d'avoir n'importe quel résultat sur le dé

In [2]:
from enum import Enum
from abc import ABC, abstractmethod

class BuildingType(Enum):
    CASTLE = "castle"
    MINE = "mine"
    WAREHOUSE = "warehouse"
    WORKSHOP = "carpeter's workshop"
    CHURCH = "church"
    MARKET = "market"
    HOUSE = "boarding house"
    BANK = "bank"
    CITYHALL = "city hall"
    WATCHTOWER = "watchtower"

class Building:
    def __init__(self, name, color=None, description="", on_place=None, on_phase_end=None, immediate_action=False):
        self.color = color
        self.name = name
        self.description = description
        self.on_place = on_place
        self.on_phase_end = on_phase_end
        self.immediate_action = immediate_action

    def trigger_on_place(self, player, game_state):
        if self.on_place:
            self.on_place(player, game_state)

    def trigger_on_phase_end(self, player, game_state):
        if self.on_phase_end:
            self.on_phase_end(player, game_state)

    

In [ ]:
def castle_effect(player):
    """
    Castle: gives one action of any kind
    """
    player.turn +=1
    player.dice = [1, 2, 3, 4, 5, 6]

def mine_effect(player):
    """
    Mine: Gives one silverling at the end of the phase (on le call à la fin du coup)
    """
    player.silverlings += 1

def warehouse_effect(player):
    """
    Warehouse: Immediately allows the player to sell one good of their choice.
    """
    if not player.goods:
        return

    # Ici on laisse choisir le mec 
    # Genre y'a une interface avec les tiles qui brillent ou whatever
    good_to_sell = player.choose("warehouse")  # là il choisit
    player.sell_good(good_to_sell)               # là ça vend

def workshop_effect(player, board):
    """
    Carpenter's Workshop: Takes the building tile of his choice (beige)
    """
    if not board.building or player.full:           # J'imagine un peu les attributs
        return
    
    building = player.choose("workshop")     # là il choisit
    player.personal_slots.append(building)

def church_effect(player, board):
    """
    Church: Takes the mine, knowledge or castle tile of his choice (c'est op wesh)
    """
    if not (board.castle and board.mine and board.knowledge) or player.full:
        return

    tile = player.choose("church")
    player.personal_slots.append(tile)

def market_effect(player, board):
    """
    Market: Allows the player to take any animal or ship.
    """
    if not (board.animal and board.ship) or player.full:
        return
    
    tile = player.choose("market")
    player.personal_slots.append(tile)

def house_effect(player, board):
    """
    Boarding House: Player takes 4 worker tiles
    """
    ...


def bank_effect(player):
    """
    Bank: Gain 2 silverlings
    """
    player.silverlings += 2


def cityhall_effect(player, game_state):
    """
    City Hall: Immediately place one additional yellow (knowledge) tile from your storage.
    """
    possible_tiles = [t for t in player.storage if t.color == "yellow"]
    if not possible_tiles:
        game_state.log(f"{player.name} has no yellow tiles to place from City Hall.")
        return
    chosen_tile = player.choose_tile_to_place(possible_tiles)
    player.place_tile(chosen_tile)
    game_state.log(f"{player.name} places {chosen_tile} using the City Hall.")


def watchtower_effect(player, game_state):
    """
    Watchtower: Gain 1 victory point for each mine in your estate.
    """
    mine_count = sum(1 for t in player.city_buildings if t.name == "Mine")
    player.victory_points += mine_count
    game_state.log(f"{player.name} gains {mine_count} VP from the Watchtower.")

In [ ]:
BUILDINGS = {
    "CASTLE": Building(
        name="Castle",
        description="Allows player to do any additional action of his choice",
        on_place=castle_effect,
        immediate_action=True
    ),
    "MINE": Building(
        name="Mine",
        description="Grants one silverling at the end of each phase",
        on_phase_end=mine_effect
    ),
    "WAREHOUSE": Building(
        name="Warehouse",
        description="Allows the player to immediately sell a good from storage",
        on_place=warehouse_effect,
        immediate_action=True
    ),
    "WORKSHOP": Building(
        name="Carpenter's Workshop",
        description="Grants one worker when placed",
        on_place=workshop_effect,
        immediate_action=True,
    ),
    "CHURCH": Building(
        name="Church",
        description="Gain VP for each different building type in the city",
        on_place=church_effect,
        immediate_action=True,
    ),
    "MARKET": Building(
        name="Market",
        description="Allows the player to sell all goods of one type",
        on_place=market_effect,
        immediate_action=True,
    ),
    "HOUSE": Building(
        name="Boarding House",
        description="Gain 4 VP when placed",
        on_place=house_effect,
        immediate_action=True,
    ),
    "BANK": Building(
        name="Bank",
        description="Gain 1 silverling for each other yellow building",
        on_place=bank_effect,
        immediate_action=True,
    ),
    "CITYHALL": Building(
        name="City Hall",
        description="Immediately place one yellow tile from storage",
        on_place=cityhall_effect,
        immediate_action=True,
    ),
    "WATCHTOWER": Building(
        name="Watchtower",
        description="Gain 1 VP for each mine in your estate",
        on_place=watchtower_effect,
        immediate_action=True,
    ),
}